### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\advai\AppData\Local\Temp\ipykernel_37572\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
#Read all pdf's inside directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf
  ✓ Loaded 10 pages

Processing: Detailed Procedure for Offset Mechanism_CCTS.pdf
  ✓ Loaded 84 pages

Total documents loaded: 94


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'file_type': 'pdf'}, page_content='METHODOLOGY\nBM FR05.001 \nAfforestation and reforestation of degraded \nmangrove habitats\nPublication Date: 27 March 2025\nVersion 1.0\nSectoral scope(s): Forestry'),
 Document(metadata={'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf

In [4]:
#Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 94 documents into 259 chunks

Example chunk:
Content: METHODOLOGY
BM FR05.001 
Afforestation and reforestation of degraded 
mangrove habitats
Publication Date: 27 March 2025
Version 1.0
Sectoral scope(s): Forestry...
Metadata: {'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'file_type': 'pdf'}, page_content='METHODOLOGY\nBM FR05.001 \nAfforestation and reforestation of degraded \nmangrove habitats\nPublication Date: 27 March 2025\nVersion 1.0\nSectoral scope(s): Forestry'),
 Document(metadata={'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf

### Embedding and VectorStoreDB 

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\VS Files\RAGpractice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}") #f.k.a get_sentence_embedding_dimension
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5958.64it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine",
                },
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 896


In [9]:
chunks

[Document(metadata={'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf', 'file_type': 'pdf'}, page_content='METHODOLOGY\nBM FR05.001 \nAfforestation and reforestation of degraded \nmangrove habitats\nPublication Date: 27 March 2025\nVersion 1.0\nSectoral scope(s): Forestry'),
 Document(metadata={'producer': 'Aspose.Words for .NET 22.4.0', 'creator': 'Microsoft Office Word', 'creationdate': '2024-12-12T14:54:00+00:00', 'author': 'Pankaj Kalyani (IN)', 'moddate': '2025-03-28T10:43:00+00:00', 'source': '..\\data\\pdf\\BM FR05.001 [Afforestation and reforestation of degraded mangrove habitats].pdf

In [10]:
#convert texts to embeddings
texts=[doc.page_content for doc in chunks]
texts

#generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 259 texts...


Batches: 100%|██████████| 9/9 [00:04<00:00,  2.13it/s]


Generated embeddings with shape: (259, 384)
Adding 259 documents to vector store...
Successfully added 259 documents to vector store
Total documents in collection: 1155


### Retriever Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Project Cycle", score_threshold=-1.0)

Retrieving documents for query: 'Project Cycle'
Top K: 5, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7ea80ee1_89',
  'content': 'B4. Project Design\nB4.1. Project Design Document Template\nB4.1.1. The non-obligated entity shall describe the proposed project activity in the Project Design \nDocument (PDD), to provide a detailed understanding of the project activity, by providing the \nfollowing information: \n i. The title, purpose(s) and objective(s) of the project activity\n ii. Unique Project Reference Number assigned to the project activity \n iii.  Roles and responsibilities, including the contact information of the non-obligated entity such \nas contact person, organization details, and other participants. \n iv. The sectoral scopes linked to the methodologies applied and relevant to the project activity.\n v. Description of how the project activity will achieve GHG emission reductions and/or removals.\n vi.  Project location covering details on organizational, physical, and geographical location of the \nproject, including information that allows to identify its uni

### Integration VectorDB Context pipeline with LLM output

In [14]:
#Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#Initialise the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

#Simple RAG function -> context retrieval + response generation
def rag_simple(query, retriever, llm, top_k=3):
    #retrieve the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"

    #answer with groq llm
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}
        
        Question:{query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [15]:
answer=rag_simple("Explain the monitoring of project implementation", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Explain the monitoring of project implementation'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 109.87it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Monitoring of project implementation**  
Monitoring is the systematic process of tracking a project’s progress against its planned objectives, ensuring that activities are carried out as described in the Project Design Document (PDD). It involves:

1. **Data collection and verification** – Gathering quantitative and qualitative information on project activities, emissions reductions, or removals, following the precision and data‑requirement guidelines (Sections 5.3 and 5.4).  
2. **Performance assessment** – Comparing actual results with the baseline and target values set in the PDD to determine whether the project is meeting its GHG‑reduction or removal goals.  
3. **Reporting and documentation** – Producing regular monitoring reports that detail activities, results, and any deviations, and storing records for audit and verification purposes.  
4. **Adaptive management** – Using monitoring findings to adjust project activities, improve efficiency, and address any issues that arise d

### Enhanced RAG Pipeline Features

In [18]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Explain the Project Registration Phase", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Explain the Project Registration Phase'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.02it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Project Registration Phase (Step 4)**  
After a project has successfully passed the Validation stage, the Accredited Carbon Verification Agency (ACVA) submits a formal registration request to the ICM Administrator. The Administrator reviews the submitted documentation—ensuring all required criteria and standards are met—and, upon approval, officially registers the project activity as an “Offset Project” on the ICM portal. This registration marks the project’s entry into the system and makes it eligible for subsequent issuance of Certified Carbon Credits (CCCs).
Sources: [{'source': 'Detailed Procedure for Offset Mechanism_CCTS.pdf', 'page': 12, 'score': 0.2078155279159546, 'preview': 'includes evaluation of a project activity on parameters as specified under the Validation and \nVerification standard of the Offset Mechanism under the ICM. Addressing comments received \nduring the public consultation period by the non-obligated entity will also form a part of the \nvalidation 

In [19]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Explain the Project Registration Phase", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Explain the Project Registration Phase'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 104.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
includes evaluation of a project activity on parameters as specified under the Validation and 
Verification standard of the Offset Mechanism under the ICM. Addressing comments received 
during the public consultation period by the non-obligated entity

 will also form a part of the 
validation step.
A3.1.4. Step 4 - Project Registration (Refer section A5): After the positive conclusion of the validation 
stage, the ACVA shall submit a request for registration for the project activity to the Administrator. 
The Administrator will evaluate the submitted documents and, upon successful review, register 
the project activity as an ‘Offset Project’.
Pre-Registration
Project Stages STEP 1
STEP 6STEP 7
STEP 2
STEP 5
STEP 3
STEP 4
Account Registration 
Verification Issuance 
Developing Project Design Document
Monitoring 
Validation 
Project Registration 
Non-Obligated Entity
Accredited Carbon 
Verification Agency (ACVA)
ICM Administrator 
Issuance RegistrationPre-Issuance
Non-Obligated Entity

includes evaluation of a project activity on parameters as specified under the Validation and 
Verification standard of the Offset Mechanism under the ICM. Addressing comments received 
during the public consultation period by the non-obligated entity w